# Extracting samples to analyse gpt annotation quality
This notebook extracts samples from the gpt labelled data created in *09_create_gpt_labelled_table_all*. These samples are used to qualitatively assess how well gpt annotates semantic classes/roles. 

## Sample for manual annotation
This code takes 1000 random rows from all the GPT labelled data for humans to manually annotate. This will be used to assess the precision, recall and F1 score of each class that GPT annotated. The gpt labelled data is available in the Estnltk google drive. 

In [ ]:
import csv
import pandas as pd
import glob
import os
import random

In [ ]:
#read all gpt labelled files in
folder = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\v07_all_types_goldstandard\\base_data\\LLM_annotated_batches"
files = glob.glob(os.path.join(folder, "*.csv"))
df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

In [ ]:
#remove duplicate rows
#before removing duplicates: 515893 rows
df = df.drop_duplicates(subset=['sentence_id', 'form'])
#after removing duplicates: 428973 rows

In [ ]:
#get random seed
random.randint(1, 100000)

In [ ]:
#take random sample of 1000 rows with the random seed
df_1000 = df.sample(n=1000, random_state = 47509)

In [ ]:
#save to csv
df_1000.to_csv("LLM_sample_1000.csv", index=False, encoding="UTF-8")

## Sample for qualitative analysis of exceptions
This code finds random 100 verb-case pairings per semantic role and level (ie, 100 verb-case pairings in the LOC n80 class, 100 in the LOC n50 class 
etc). Then, for each of these verb-case pairing, it extracts a random sample of 100 sentences where the verb's dependent isn't in the dominant semantic class (ie for a verb-case pairing in the LOC n80 class, it extracts 100 sentences where the verb's dependent in that case was not LOC). These will be manually analysed by a linguist to find the main reason for why the exceptions occur.   

### verb-case pairs

In [ ]:
import csv
import sqlite3
import pandas as pd
import random

In [ ]:
# put all n80 and n50 verb-case pairs into a database table
csv_file = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\v07_all_types_goldstandard\\base_data\\verb_case_all.csv"
DB_FILE = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_spatial_obl_2026_04_22.db"
df_csv = pd.read_csv(csv_file, delimiter=';')

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df_csv.to_sql('verb_case_levels', conn, if_exists="replace", index=False)

conn.close()

In [ ]:
#get ELT class verb-case pairings that aren't in L, T or E n80 and n50
def get_ELT_verbs(db_file, level1, level2):
    #connect to database
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    #read counts into dataframe
    df = pd.read_sql(f"""
        SELECT verb, verb_compound, morph_case, level
        FROM gpt_ELT_n_class_wide as n
        WHERE n.level IN ('{level1}','{level2}')
        AND (verb, verb_compound, morph_case) NOT IN (
          SELECT verb, verb_compound, morph_case
          FROM verb_case_levels)
    """, conn)

    # Commit and close
    conn.close()

    return df

In [ ]:
#take verb-case pairing in n50 and n80 classes
df_L_n50 = df_csv.loc[(df_csv["level"] == 'n50') & (df_csv["tag"] == 'L')]
df_A_n50 = df_csv.loc[(df_csv["level"] == 'n50') & (df_csv["tag"] == 'A')]
df_E_n50 = df_csv.loc[(df_csv["level"] == 'n50') & (df_csv["tag"] == 'E')]
df_ELT_n80 = get_ELT_verbs(DB_FILE, 'n80', 'n90')
df_ELT_n50 = get_ELT_verbs(DB_FILE, 'n70', 'n30')

In [ ]:
#random seed for extracting random verb-case pairings
random.randint(1, 100000)

In [ ]:
#extract random sample of 100 verb-case pairings
df_L_n50_100 = df_L_n50.sample(n=100, random_state = 83186)
df_A_n50_100 = df_A_n50.sample(n=100, random_state = 83186)
df_E_n50_100 = df_E_n50.sample(n=100, random_state = 83186)
df_ELT_n80_100 = df_ELT_n80.sample(n=100, random_state = 83186)
df_ELT_n50_100 = df_ELT_n50.sample(n=100, random_state = 83186)

In [ ]:
#put all the verb-case pairings into the same database table
df_merged = pd.concat([df_L_n50_100, df_A_n50_100, df_E_n50_100, df_ELT_n50_100, df_ELT_n80_100], ignore_index=True)

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df_merged.to_sql('verb_case_sample_100', conn, if_exists="replace", index=False)

conn.close()

### sentences

In [ ]:
#extract all sentences for each verb-case pairing where the nominal adverbial isn't in the dominant semantic class
def verb_case_sentences(verb, verb_compound, case, tag, db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    if pd.isna(verb_compound):
        df = pd.read_sql(f"""
            SELECT sentence_id, head_id, head_loc, verb, verb_compound, morph_case, lemma, form, sentence, timex_tag, ekilex_tag, ner_tag, tag, id
            FROM gpt_labelled_clean2 as n
            WHERE verb = '{verb}' and verb_compound = '' and morph_case = '{case}' and (tag != '{tag}' OR tag IS NULL)
        """, conn)
    else:
        df = pd.read_sql(f"""
            SELECT sentence_id, head_id, head_loc, verb, verb_compound, morph_case, lemma, form, sentence, timex_tag, ekilex_tag, ner_tag, tag, id
            FROM gpt_labelled_clean2 as n
            WHERE verb = '{verb}' and verb_compound = '{verb_compound}' and morph_case = '{case}' and (tag != '{tag}' OR tag IS NULL)
        """, conn)

    # Commit and close
    conn.close()

    return df

In [ ]:
#read files in
csv_file = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\v07_all_types_goldstandard\\base_data\\verb_case_sample.csv"
DB_FILE = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_spatial_obl_2026_04_22.db"
df_verb_cases = pd.read_csv(csv_file, delimiter=';')

#extract sentences
df_list = []
for index, row in df_verb_cases.iterrows():
    df_sentences = verb_case_sentences(row['verb'], row['verb_compound'], row['morph_case'], row['tag'], DB_FILE)
    df_list.append(df_sentences)

In [ ]:
#put all sentences into the same dataframe and save as csv
df_merged_sen = pd.concat(df_list, ignore_index=True)
df_merged_sen.to_csv("random_sample_sentences.csv", sep=';', encoding='utf-8', index=False)

## Sample for prompt development
This code takes examples of wrongly annotated sentences from the qualitative sample and shuffles them. These are used to check whether new prompts can remove these mistakes.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel("loc_role_devset.xlsx")
df2 = pd.concat(df, ignore_index=True)
shuffle = df.sample(frac=1)

In [ ]:
#for checking save id, sentence, word and correct class column
shuffle.to_csv("devset_loc_role.csv", index=True, encoding="UTF-8", sep = ";")
#for feeding to gpt leave out the id and correct annotation column
shuffle.to_csv("devset_loc_role_for_gpt.csv", index=False, encoding="UTF-8", sep = ";", columns = ['sentence', 'word'])